# NOTEBOOK : 03_silver_transform
# PURPOSE  : Build full Snowflake schema in Silver layer
#            7 dimension tables + 3 fact tables
# SILVER SCHEMA : customer360_silver

# DIMENSIONS
#   dim_date            — full date spine 2023–2026
#   dim_city            — city, state, region, tier
#   dim_location        — city_key FK + zone + pincode
#   dim_category        — category, department, super_category
#   dim_product         — product_key, category_key FK, brand, price_tier
#   dim_payment_method  — method, provider, type, digital flag
#   dim_customer        — one row per customer (SCD0, no history)
#
# FACTS
#   fact_transactions   — one row per transaction
#   fact_sessions       — one row per app session
#   fact_support        — one row per support ticket

 

# 03 · Silver transform — Snowflake schema
#  Reads all 4 Bronze tables and builds 7 dimensions + 3 facts.
# **Always runs after 02_incremental_generator.**

## Cell 1 — Install & restart

In [0]:
%pip install faker --quiet
dbutils.library.restartPython()

## Cell 2 — Imports & config

In [0]:
import uuid
import random
from datetime import date, datetime, timedelta
 
from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType, BooleanType
)
 
fake  = Faker("en_IN")
spark = SparkSession.builder.getOrCreate()
 
RUN_TS = datetime.utcnow()
RUN_ID = RUN_TS.strftime("%Y%m%d_%H%M%S")
 
BRONZE = "customer360"
SILVER = "customer360_silver"
 
print(f"Run ID  : {RUN_ID}")
print(f"Bronze  : {BRONZE}")
print(f"Silver  : {SILVER}")

## Cell 3 — Create Silver schema

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {SILVER}
    COMMENT 'Customer 360 — Silver layer (Snowflake schema)'
""")
 
spark.sql(f"SHOW SCHEMAS").filter(
    F.col("databaseName").contains("customer360")
).show()

## Cell 4 — `dim_date`
Generated programmatically — no source table needed.
Covers 2023-01-01 → 2026-12-31 (full history + 2 years forward).
This is standard practice: pre-build the date spine so every
fact table can join to it without DATE_TRUNC at query time.
 

In [0]:
date_rows = []
current   = date(2023, 1, 1)
end_date  = date(2026, 12, 31)
 
MONTH_NAMES = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
 
# Indian public holidays (simplified — extend as needed)
IN_HOLIDAYS = {
    date(2023, 1, 26), date(2023, 8, 15), date(2023, 10, 2),
    date(2024, 1, 26), date(2024, 8, 15), date(2024, 10, 2),
    date(2025, 1, 26), date(2025, 8, 15), date(2025, 10, 2),
    date(2026, 1, 26), date(2026, 8, 15), date(2026, 10, 2),
}
 
while current <= end_date:
    week_num  = current.isocalendar()[1]
    quarter   = (current.month - 1) // 3 + 1
    is_wkend  = 1 if current.weekday() >= 5 else 0
    is_hol    = 1 if current in IN_HOLIDAYS else 0
 
    date_rows.append((
        current.strftime("%Y-%m-%d"),          # date_key (PK, string)
        current,                               # full_date
        current.day,                           # day_of_month
        current.weekday() + 1,                 # day_of_week (1=Mon)
        current.strftime("%A"),                # day_name
        week_num,                              # week_number
        current.month,                         # month
        MONTH_NAMES[current.month - 1],        # month_name
        quarter,                               # quarter
        f"Q{quarter}",                         # quarter_label
        current.year,                          # year
        f"{current.year}-Q{quarter}",          # fiscal_quarter_label
        is_wkend,                              # is_weekend
        is_hol,                                # is_holiday
        1 if (is_wkend or is_hol) else 0,      # is_non_working_day
    ))
    current += timedelta(days=1)
 
dim_date_schema = StructType([
    StructField("date_key",              StringType(),  False),
    StructField("full_date",             DateType(),    True),
    StructField("day_of_month",          IntegerType(), True),
    StructField("day_of_week",           IntegerType(), True),
    StructField("day_name",              StringType(),  True),
    StructField("week_number",           IntegerType(), True),
    StructField("month",                 IntegerType(), True),
    StructField("month_name",            StringType(),  True),
    StructField("quarter",               IntegerType(), True),
    StructField("quarter_label",         StringType(),  True),
    StructField("year",                  IntegerType(), True),
    StructField("fiscal_quarter_label",  StringType(),  True),
    StructField("is_weekend",            IntegerType(), True),
    StructField("is_holiday",            IntegerType(), True),
    StructField("is_non_working_day",    IntegerType(), True),
])
 
(
    spark.createDataFrame(date_rows, schema=dim_date_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_date")
)
 
count = spark.table(f"{SILVER}.dim_date").count()
print(f"{SILVER}.dim_date — {count} rows  ({date(2023,1,1)} → {end_date})")

## Cell 5 — `dim_city`
Snowflake normalization level 1: city master table.
dim_location will FK into this — avoids repeating
city name, state, region in every location row.

In [0]:
# Master city reference — state, region, tier classification
CITY_MASTER = {
    "Mumbai":    ("Maharashtra",    "West",  "Tier 1"),
    "Delhi":     ("Delhi",          "North", "Tier 1"),
    "Bangalore": ("Karnataka",      "South", "Tier 1"),
    "Chennai":   ("Tamil Nadu",     "South", "Tier 1"),
    "Hyderabad": ("Telangana",      "South", "Tier 1"),
    "Pune":      ("Maharashtra",    "West",  "Tier 2"),
    "Kolkata":   ("West Bengal",    "East",  "Tier 1"),
    "Ahmedabad": ("Gujarat",        "West",  "Tier 2"),
    "Jaipur":    ("Rajasthan",      "North", "Tier 2"),
    "Coimbatore":("Tamil Nadu",     "South", "Tier 2"),
}
 
city_rows = []
for city, (state, region, tier) in CITY_MASTER.items():
    city_rows.append((
        "CITY_" + city[:3].upper(),   # city_key
        city,                          # city_name
        state,
        region,
        tier,
    ))
 
dim_city_schema = StructType([
    StructField("city_key",   StringType(), False),
    StructField("city_name",  StringType(), True),
    StructField("state",      StringType(), True),
    StructField("region",     StringType(), True),
    StructField("city_tier",  StringType(), True),
])
 
(
    spark.createDataFrame(city_rows, schema=dim_city_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_city")
)
 
print(f"{SILVER}.dim_city — {len(city_rows)} rows")

## Cell 6 — `dim_location`

Snowflake normalization level 2: location table.
Each location = one city + one pincode + one zone.
Multiple locations can map to the same city.
fact_transactions and fact_sessions FK into this.

In [0]:
ZONES = ["North Zone","South Zone","East Zone","West Zone","Central Zone"]
 
# Generate 3 locations per city (different pincodes + zones)
location_rows = []
for city, (state, region, tier) in CITY_MASTER.items():
    city_key = "CITY_" + city[:3].upper()
    for i in range(1, 4):
        pincode      = str(random.randint(400000, 799999))
        location_key = f"LOC_{city[:3].upper()}_{i:02d}"
        zone         = random.choice(ZONES)
        location_rows.append((
            location_key,
            city_key,
            city,          # denormalised city_name for convenience
            pincode,
            zone,
        ))
 
dim_location_schema = StructType([
    StructField("location_key", StringType(), False),
    StructField("city_key",     StringType(), True),
    StructField("city_name",    StringType(), True),
    StructField("pincode",      StringType(), True),
    StructField("zone",         StringType(), True),
])
 
(
    spark.createDataFrame(location_rows, schema=dim_location_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_location")
)
 
# Build a lookup dict: city_name → list of location_keys
# Used later when assigning location_key to fact rows
city_to_locations = {}
for r in location_rows:
    city_to_locations.setdefault(r[2], []).append(r[0])
 
print(f"{SILVER}.dim_location — {len(location_rows)} rows")

## Cell 7 — `dim_category`

Snowflake normalization: category master.
dim_product will FK into this

In [0]:
CATEGORY_MASTER = {
    "Electronics":   ("Technology",   "Hard Goods"),
    "Fashion":        ("Apparel",      "Soft Goods"),
    "Groceries":      ("Food",         "Consumables"),
    "Travel":         ("Services",     "Experiences"),
    "Entertainment":  ("Media",        "Experiences"),
    "Health":         ("Wellness",     "Consumables"),
    "Sports":         ("Active Life",  "Hard Goods"),
}
 
category_rows = []
for cat, (dept, super_cat) in CATEGORY_MASTER.items():
    category_rows.append((
        "CAT_" + cat[:4].upper(),
        cat,
        dept,
        super_cat,
    ))
 
dim_category_schema = StructType([
    StructField("category_key",    StringType(), False),
    StructField("category_name",   StringType(), True),
    StructField("department",      StringType(), True),
    StructField("super_category",  StringType(), True),
])
 
(
    spark.createDataFrame(category_rows, schema=dim_category_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_category")
)
 
# Lookup: category_name → category_key
cat_to_key = {r[1]: r[0] for r in category_rows}
 
print(f"{SILVER}.dim_category — {len(category_rows)} rows")


## Cell 8 — `dim_product`

Built from distinct product_ids in Bronze transactions.
Each product gets a category_key FK → dim_category.
 

In [0]:
BRANDS = {
    "Electronics":  ["Samsung","Apple","Sony","OnePlus","LG"],
    "Fashion":       ["Zara","H&M","Myntra","Fabindia","Levis"],
    "Groceries":     ["BigBasket","DMart","Reliance Fresh","Spencer","More"],
    "Travel":        ["MakeMyTrip","Goibibo","IRCTC","Yatra","EaseMyTrip"],
    "Entertainment": ["BookMyShow","Hotstar","Netflix","ZEE5","SonyLiv"],
    "Health":        ["Apollo","Medplus","1mg","PharmEasy","Healthkart"],
    "Sports":        ["Decathlon","Nike","Adidas","Puma","Reebok"],
}
 
PRICE_TIERS = ["Budget","Mid-range","Premium","Luxury"]
 
# Read distinct product_ids + categories from Bronze
bronze_products = (
    spark.table(f"{BRONZE}.bronze_transactions")
    .select("product_id", "category")
    .distinct()
    .collect()
)
 
product_rows = []
seen_products = set()
 
for row in bronze_products:
    pid = row.product_id
    cat = row.category
    if pid in seen_products:
        continue
    seen_products.add(pid)
 
    cat_key    = cat_to_key.get(cat, "CAT_UNKN")
    brand      = random.choice(BRANDS.get(cat, ["Generic"]))
    price_tier = random.choice(PRICE_TIERS)
    base_price = round(random.choices(
        [random.uniform(50, 500),
         random.uniform(500, 5000),
         random.uniform(5000, 50000)],
        weights=[50, 35, 15]
    )[0], 2)
 
    product_rows.append((
        pid,          # product_key = product_id from Bronze
        cat_key,
        cat,          # denormalised for convenience
        brand,
        price_tier,
        base_price,
    ))
 
dim_product_schema = StructType([
    StructField("product_key",   StringType(), False),
    StructField("category_key",  StringType(), True),
    StructField("category_name", StringType(), True),
    StructField("brand",         StringType(), True),
    StructField("price_tier",    StringType(), True),
    StructField("base_price",    DoubleType(), True),
])
 
(
    spark.createDataFrame(product_rows, schema=dim_product_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_product")
)
 
print(f"{SILVER}.dim_product — {len(product_rows)} rows")

## Cell 9 — `dim_payment_method`

In [0]:
PAYMENT_MASTER = [
    ("Credit Card",  "Visa/Mastercard", "Card",       0, 1),
    ("Debit Card",   "Visa/Mastercard", "Card",       0, 0),
    ("UPI",          "NPCI",            "Digital",    1, 0),
    ("Net Banking",  "Bank",            "Digital",    1, 0),
    ("Wallet",       "Paytm/PhonePe",   "Digital",    1, 0),
]
 
payment_rows = []
for method, provider, ptype, is_digital, is_emi in PAYMENT_MASTER:
    payment_rows.append((
        "PAY_" + method.replace(" ","_").upper()[:8],
        method,
        provider,
        ptype,
        is_digital,
        is_emi,
    ))
 
dim_payment_schema = StructType([
    StructField("payment_key",     StringType(),  False),
    StructField("method_name",     StringType(),  True),
    StructField("provider",        StringType(),  True),
    StructField("payment_type",    StringType(),  True),
    StructField("is_digital",      IntegerType(), True),
    StructField("is_emi_eligible", IntegerType(), True),
])
 
(
    spark.createDataFrame(payment_rows, schema=dim_payment_schema)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_payment_method")
)
 
# Lookup: method_name → payment_key
pay_to_key = {r[1]: r[0] for r in payment_rows}
 
print(f"{SILVER}.dim_payment_method — {len(payment_rows)} rows")

## Cell 10 — `dim_customer`

One row per customer — sourced from bronze_users.
Uses DISTINCT + dedup on user_id to handle the fact
that the same user_id can appear in multiple pipeline runs
(NB02 appends, so user_id is unique but rows may duplicate).

Surrogate key pattern: customer_key = "CUST_" + user_id.
Natural key: user_id (from Bronze source).

In [0]:
from pyspark.sql.window import Window
 
users_raw = spark.table(f"{BRONZE}.bronze_users")
 
# Keep latest record per user_id (in case NB02 re-inserted same user)
window_latest = Window.partitionBy("user_id").orderBy(
    F.col("pipeline_run_id").desc()
)
 
dim_customer = (
    users_raw
    .withColumn("row_num", F.row_number().over(window_latest))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .withColumn("customer_key",
        F.concat(F.lit("CUST_"), F.col("user_id")))
    .withColumn("signup_date",
        F.to_date("signup_date"))
    .withColumn("customer_age_days",
        F.datediff(F.current_date(), F.col("signup_date")))
    .select(
        "customer_key",
        "user_id",
        "name",
        "age",
        "gender",
        F.col("signup_date"),
        "customer_age_days",
        "city",
        "country",
        "email",
        "segment",
        "is_active",
        "pipeline_run_id",
    )
)
 
(
    dim_customer
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.dim_customer")
)
 
count = spark.table(f"{SILVER}.dim_customer").count()
print(f"{SILVER}.dim_customer — {count} rows")

## Cell 11 — `fact_transactions`

One row per transaction from Bronze.
Enriches each row with 5 dimension keys:
customer_key, product_key, date_key,
location_key, payment_key

Also computes net_amount (after discount).

In [0]:
# Build city → location_key mapping as a Spark DataFrame for join
location_lookup = (
    spark.table(f"{SILVER}.dim_location")
    .select("location_key", "city_name")
    .groupBy("city_name")
    .agg(F.first("location_key").alias("location_key"))
)
 
# Build payment method lookup
payment_lookup = (
    spark.table(f"{SILVER}.dim_payment_method")
    .select(
        F.col("method_name").alias("payment_method"),
        F.col("payment_key")
    )
)
 
bronze_txns = (
    spark.table(f"{BRONZE}.bronze_transactions")
    .withColumn("transaction_timestamp",
        F.to_timestamp("transaction_timestamp"))
    .withColumn("date_key",
        F.date_format("transaction_timestamp", "yyyy-MM-dd"))
    .withColumn("customer_key",
        F.concat(F.lit("CUST_"), F.col("user_id")))
    # net_amount = amount after discount
    .withColumn("net_amount",
        F.round(
            F.col("amount") * (1 - F.col("discount_pct") / 100.0),
        2))
)
 
fact_transactions = (
    bronze_txns
    # Join product key — product_id in Bronze = product_key in dim_product
    .withColumnRenamed("product_id", "product_key")
    # Join location key
    .join(
        location_lookup.withColumnRenamed("city_name", "city"),
        on="city", how="left"
    )
    # Join payment key
    .join(payment_lookup, on="payment_method", how="left")
    .select(
        F.col("transaction_id").alias("transaction_key"),
        "customer_key",
        "product_key",
        "date_key",
        F.coalesce("location_key", F.lit("LOC_UNKNOWN")).alias("location_key"),
        F.coalesce("payment_key",  F.lit("PAY_UNKNOWN")).alias("payment_key"),
        F.col("amount").alias("gross_amount"),
        "net_amount",
        "discount_pct",
        "status",
        "platform",
        "category",
        "pipeline_run_id",
    )
    # Remove duplicate transaction_keys (safety dedup)
    .dropDuplicates(["transaction_key"])
)
 
(
    fact_transactions
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.fact_transactions")
)
 
count = spark.table(f"{SILVER}.fact_transactions").count()
print(f"{SILVER}.fact_transactions — {count} rows")

## Cell 12 — `fact_sessions`

In [0]:
bronze_sessions = (
    spark.table(f"{BRONZE}.bronze_app_usage")
    .withColumn("session_start",
        F.to_timestamp("session_start"))
    .withColumn("session_end",
        F.to_timestamp("session_end"))
    .withColumn("date_key",
        F.date_format("session_start", "yyyy-MM-dd"))
    .withColumn("customer_key",
        F.concat(F.lit("CUST_"), F.col("user_id")))
)
 
# Sessions don't have city in Bronze — join via dim_customer city
customer_city = (
    spark.table(f"{SILVER}.dim_customer")
    .select("customer_key",
            F.col("city").alias("customer_city"))
)
 
fact_sessions = (
    bronze_sessions
    .join(customer_city, on="customer_key", how="left")
    .join(
        location_lookup.withColumnRenamed("city_name", "customer_city"),
        on="customer_city", how="left"
    )
    .select(
        F.col("session_id").alias("session_key"),
        "customer_key",
        "date_key",
        F.coalesce("location_key", F.lit("LOC_UNKNOWN")).alias("location_key"),
        "session_duration_mins",
        "pages_visited",
        "actions_taken",
        "device_type",
        "platform",
        "is_bounce",
        "pipeline_run_id",
    )
    .dropDuplicates(["session_key"])
)
 
(
    fact_sessions
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.fact_sessions")
)
 
count = spark.table(f"{SILVER}.fact_sessions").count()
print(f"{SILVER}.fact_sessions — {count} rows")

## Cell 13 — `fact_support`

In [0]:
bronze_tickets = (
    spark.table(f"{BRONZE}.bronze_support_tickets")
    .withColumn("created_at",
        F.to_timestamp("created_at"))
    .withColumn("date_key",
        F.date_format("created_at", "yyyy-MM-dd"))
    .withColumn("customer_key",
        F.concat(F.lit("CUST_"), F.col("user_id")))
    .withColumn("is_resolved",
        (F.col("status") == "closed").cast("int"))
)
 
fact_support = (
    bronze_tickets
    .select(
        F.col("ticket_id").alias("ticket_key"),
        "customer_key",
        "date_key",
        "issue_type",
        "priority",
        "status",
        "is_resolved",
        "resolution_hours",
        "satisfaction_score",
        "pipeline_run_id",
    )
    .dropDuplicates(["ticket_key"])
)
 
(
    fact_support
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.fact_support")
)
 
count = spark.table(f"{SILVER}.fact_support").count()
print(f"{SILVER}.fact_support — {count} rows")

## Cell 14 — Verify all 8 Silver tables

 

In [0]:
%sql
SHOW TABLES IN customer360_silver;

In [0]:
SILVER_TABLES = [
    "dim_date",
    "dim_city",
    "dim_location",
    "dim_category",
    "dim_product",
    "dim_payment_method",
    "dim_customer",
    "fact_transactions",
    "fact_sessions",
    "fact_support",
]
 
print("=" * 60)
print(f"  Silver transform complete | RUN_ID: {RUN_ID}")
print("=" * 60)
for t in SILVER_TABLES:
    df    = spark.table(f"{SILVER}.{t}")
    rows  = df.count()
    cols  = len(df.columns)
    layer = "DIM" if t.startswith("dim") else "FACT"
    print(f"  [{layer}] {t:<30} {rows:>7} rows  {cols:>2} cols")
print("=" * 60)

## Cell 15 — Spot-check joins are clean
 

 

In [0]:
%sql
-- Verify fact_transactions → dim_customer join integrity
-- Should return 0 orphan rows
SELECT COUNT(*) AS orphan_transactions
FROM customer360_silver.fact_transactions f
LEFT JOIN customer360_silver.dim_customer c
ON f.customer_key = c.customer_key
WHERE c.customer_key IS NULL;

In [0]:
%sql
-- Revenue by city tier + quarter — the kind of query
-- that is only possible with a proper snowflake schema
SELECT
dc.city_tier,
dd.fiscal_quarter_label,
COUNT(ft.transaction_key)     AS transactions,
ROUND(SUM(ft.net_amount), 0)  AS net_revenue,
ROUND(AVG(ft.gross_amount), 2) AS avg_order_value
FROM customer360_silver.fact_transactions ft
JOIN customer360_silver.dim_location   dl ON ft.location_key = dl.location_key
JOIN customer360_silver.dim_city       dc ON dl.city_key      = dc.city_key
JOIN customer360_silver.dim_date       dd ON ft.date_key      = dd.date_key
GROUP BY dc.city_tier, dd.fiscal_quarter_label
ORDER BY dc.city_tier, dd.fiscal_quarter_label;
 

 


In [0]:
%sql

-- Payment method breakdown — digital vs physical revenue split
SELECT
dp.payment_type,
dp.is_digital,
COUNT(*)                       AS txn_count,
ROUND(SUM(ft.net_amount), 0)   AS net_revenue,
ROUND(AVG(ft.net_amount), 2)   AS avg_txn_value
FROM customer360_silver.fact_transactions ft
JOIN customer360_silver.dim_payment_method dp
ON ft.payment_key = dp.payment_key
GROUP BY dp.payment_type, dp.is_digital
ORDER BY net_revenue DESC;

In [0]:

print("Notebook 03 complete.")
print("Next: run 04_gold_metrics to compute LTV, RFM, churn flag.")
dbutils.notebook.exit(RUN_ID)